# 02: Preprocessing (Phase 2)

**Goal:** one tidy table with one row per company per label year and one column per financial statement item, in USD millions, with every cleaning decision visible.

**Inputs:** `data/interim/filings_3674.csv` and `data/interim/num_3674.parquet` (Phase 1), plus SEC company facts for the four case companies in `data/raw/companyfacts/`.

**Outputs:**
- `data/processed/statements_clean.csv`: the clean statements table (label years 2014 to 2025; 2014 is only the base year for growth ratios)
- `data/processed/tag_sources.csv`: which XBRL tag (or rule) supplied every value
- `data/processed/preprocessing_steps.csv`: rows remaining after each step
- `data/interim/dropped_rows.csv`: every dropped company-year with the reason

All functions are in `src/preprocess.py`.

In [1]:
import sys
import pandas as pd

sys.path.insert(0, "..")
from src import config
from src import preprocess as pp
from src.load_sec import download_companyfacts, annual_facts_for_filings, sec_user_agent

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", "{:,.1f}".format)

filings_all = pd.read_csv(config.INTERIM_DIR / "filings_3674.csv", parse_dates=["period", "filed", "accepted"])
num = pd.read_parquet(config.INTERIM_DIR / "num_3674.parquet")
steps = [("Phase 1 annual filings (all label years)", len(filings_all), filings_all["cik"].nunique())]
print(f"Phase 1 filings: {len(filings_all):,} | num rows: {len(num):,}")

Phase 1 filings: 1,128 | num rows: 491,614


## Step 1: Year range and duplicate filings

Keep label years 2014 to 2025 (2014 only as the growth base year; earlier and later years are partial edges of the download). Then, where a company has more than one annual filing for the same fiscal year-end (an original 10-K plus a 10-K/A, or a re-filed 10-K), keep the **most recently filed** one, because it carries the corrected figures (hard rule 4).

In [2]:
in_range = filings_all[filings_all["label_year"].between(config.GROWTH_BASE_YEAR, config.LABEL_YEAR_END)]
steps.append((f"Label years {config.GROWTH_BASE_YEAR}-{config.LABEL_YEAR_END}", len(in_range), in_range["cik"].nunique()))
print(f"Dropped as outside {config.GROWTH_BASE_YEAR}-{config.LABEL_YEAR_END}: {len(filings_all) - len(in_range)}")

filings, dups = pp.select_filings(filings_all, config.GROWTH_BASE_YEAR, config.LABEL_YEAR_END)
steps.append(("Deduplicated (latest filing per company-period)", len(filings), filings["cik"].nunique()))
print(f"Duplicate filings dropped: {len(dups)} ({dups['form'].value_counts().to_dict()})")
print("Exactly one filing per company per label year:", not filings.duplicated(["cik", "label_year"]).any())
print("Forms kept:", filings["form"].value_counts().to_dict())

Dropped as outside 2014-2025: 14
Duplicate filings dropped: 27 ({'10-K': 26, '10-K/A': 1})
Exactly one filing per company per label year: True
Forms kept: {'10-K': 1058, '10-K/A': 29}


## Step 2: Keep only the company's own consolidated, full-year USD values

Hard rule 3 (extended in Phase 1 to also require an empty `segments` field):
`ddate` = the filing's period, `uom` = USD, no `coreg`, no `segments`, `qtrs` 0 (balance sheet) or 4 (full year), and a non-empty value.

In [3]:
num_sel = num[num["adsh"].isin(filings["adsh"])]
values = pp.consolidated_values(num, filings)
print(f"num rows of the selected filings: {len(num_sel):,}")
print(f"rows kept by the value filter:    {len(values):,} ({len(values) / len(num_sel):.1%})")
print("One value per filing, tag and qtrs:", not values.duplicated(["adsh", "tag", "qtrs"]).any())

num rows of the selected filings: 474,830
rows kept by the value filter:    99,251 (20.9%)
One value per filing, tag and qtrs: True


## Step 3: Pick one value per statement item (tag priority lists)

Companies use different XBRL tags for the same item, and switch over time (e.g. `SalesRevenueNet` was replaced by `RevenueFromContractWithCustomerExcludingAssessedTax` under the ASC 606 revenue standard). Each item has a priority list; the first tag a filing reports wins, and the tag is recorded.

In [4]:
for item, (qtrs, tags) in pp.ITEMS.items():
    print(f"{item:24s} qtrs={qtrs}  {' > '.join(tags)}")

total_assets             qtrs=0  Assets
current_assets           qtrs=0  AssetsCurrent
current_liabilities      qtrs=0  LiabilitiesCurrent
inventory                qtrs=0  InventoryNet
cash                     qtrs=0  CashAndCashEquivalentsAtCarryingValue > Cash > CashCashEquivalentsRestrictedCashAndRestrictedCashEquivalents
total_liabilities        qtrs=0  Liabilities
liabilities_and_equity   qtrs=0  LiabilitiesAndStockholdersEquity
equity                   qtrs=0  StockholdersEquity
equity_incl_nci          qtrs=0  StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest
noncontrolling_interest  qtrs=0  MinorityInterest
temporary_equity         qtrs=0  TemporaryEquityCarryingAmountIncludingPortionAttributableToNoncontrollingInterests > TemporaryEquityCarryingAmountAttributableToParent > RedeemableNoncontrollingInterestEquityCarryingAmount
retained_earnings        qtrs=0  RetainedEarningsAccumulatedDeficit
revenue                  qtrs=4  Revenues > RevenueFromContractWi

In [5]:
wide, sources = pp.pick_items(values)
no_usd = filings[~filings["adsh"].isin(wide.index)]
print(f"Filings with at least one item: {len(wide):,}")
print(f"Filings with no USD statement values at all: {len(no_usd)}")
no_usd_units = num[num["adsh"].isin(no_usd["adsh"])].groupby("adsh")["uom"].agg(lambda s: s.value_counts().index[0])
no_usd.assign(main_unit=no_usd["adsh"].map(no_usd_units))[["cik", "name", "label_year", "form", "main_unit"]]

Filings with at least one item: 1,085
Filings with no USD statement values at all: 2


,cik,name,label_year,form,main_unit
932,1596089,ETERIS B.V.,2014,10-K,EUR
1063,1840416,SONO GROUP N.V.,2024,10-K,EUR


In [6]:
# How often each tag supplied each item (number of filings).
tag_usage = (sources.melt(var_name="item", value_name="tag").dropna()
             .groupby(["item", "tag"]).size().rename("filings").reset_index())
tag_usage

,item,tag,filings
0,cash,Cash,39
1,cash,CashAndCashEquivalentsAtCarryingValue,1006
2,cash,CashCashEquivalentsRestrictedCashAndRestricted...,24
3,cost_of_revenue,CostOfGoodsAndServicesSold,446
4,cost_of_revenue,CostOfGoodsSold,124
5,cost_of_revenue,CostOfRevenue,442
6,current_assets,AssetsCurrent,1075
7,current_liabilities,LiabilitiesCurrent,1071
8,equity,StockholdersEquity,1048
9,equity_incl_nci,StockholdersEquityIncludingPortionAttributable...,285


## Step 4: Case-company interest expense from SEC company facts

The quarterly `num` files only contain facts shown on the primary statements. Intel and Wolfspeed disclose interest expense only in the notes, so it is missing there. The SEC company facts API returns every us-gaap fact from the same 10-K filings, including the notes, tagged with the filing's accession number. Only values reported **in that exact filing**, covering a full year and ending at the filing's period, are used, and only to fill gaps.

In [7]:
case_filings = filings[filings["cik"].isin(config.CASE_COMPANIES)]
missing_files = [c for c in config.CASE_COMPANIES if not (config.COMPANYFACTS_DIR / f"CIK{c:010d}.json").exists()]
if missing_files:
    download_companyfacts(missing_files, config.COMPANYFACTS_DIR, sec_user_agent(config.SEC_USER_AGENT_ENV))
print("Company facts files:", sorted(p.name for p in config.COMPANYFACTS_DIR.glob("*.json")))

case_facts = annual_facts_for_filings(config.COMPANYFACTS_DIR, case_filings, pp.ITEMS["interest_expense"][1])
print("Conflicting values for the same filing and tag:", case_facts.duplicated(["adsh", "tag"]).sum())

before = sources.loc[sources.index.intersection(case_filings["adsh"]), "interest_expense"].copy()
n_filled = pp.fill_case_interest(wide, sources, case_facts)
print(f"Case-company interest expense values filled from company facts: {n_filled}")

filled = case_filings.set_index("adsh").loc[before.index[before.isna()], ["cik", "label_year"]]
filled["company"] = filled["cik"].map(config.CASE_COMPANIES)
filled["interest_expense_musd"] = wide.loc[filled.index, "interest_expense"] / 1e6
filled["source"] = sources.loc[filled.index, "interest_expense"]
filled.sort_values(["company", "label_year"])

Company facts files: ['CIK0000002488.json', 'CIK0000050863.json', 'CIK0000895419.json', 'CIK0001045810.json']
Conflicting values for the same filing and tag: 0
Case-company interest expense values filled from company facts: 17


,cik,label_year,company,interest_expense_musd,source
adsh,,,,,
0000050863-15-000015,50863,2014,Intel,192.0,companyfacts:InterestExpense
0000050863-16-000105,50863,2015,Intel,337.0,companyfacts:InterestExpense
0000050863-17-000012,50863,2016,Intel,733.0,companyfacts:InterestExpense
0000050863-18-000007,50863,2017,Intel,646.0,companyfacts:InterestExpense
0000050863-19-000007,50863,2018,Intel,468.0,companyfacts:InterestExpense
0000050863-20-000011,50863,2019,Intel,489.0,companyfacts:InterestExpense
0000050863-21-000010,50863,2020,Intel,629.0,companyfacts:InterestExpense
0000050863-22-000007,50863,2021,Intel,597.0,companyfacts:InterestExpense
0000050863-23-000006,50863,2022,Intel,496.0,companyfacts:InterestExpense


## Step 5: Derived items, missing-value rules and USD millions

In order (see `derive_and_flag` in `src/preprocess.py`):
- **Total liabilities** missing: total liabilities and equity − temporary (mezzanine) equity − total equity incl. non-controlling interests.
- **Equity** (parent) missing but total equity incl. NCI reported: use the total, flagged `equity_includes_nci`.
- **Gross profit** missing: revenue − cost of revenue (reported gross profit is always preferred).
- **Interest expense** missing: cash interest paid as a flagged substitute; still missing and no debt reported: 0; otherwise left missing (coverage "not available").
- **Inventory**: 0 only for companies that never report inventory in any year; otherwise left missing.
- All money converted to **USD millions**; `low_revenue` flags revenue below USD 10 million (kept, not dropped).

In [8]:
debt = pp.has_debt(values)
statements, sources = pp.derive_and_flag(wide, sources, filings, debt)
steps.append(("With USD statement values", len(statements), statements["cik"].nunique()))

flags = ["total_liabilities_derived", "equity_includes_nci", "gross_profit_derived", "inventory_assumed_zero",
         "cash_includes_restricted", "net_income_from_profitloss", "low_revenue"]
print(statements[flags].sum().rename("rows flagged").to_string())
print()
print("Interest expense source:")
print(statements["interest_source"].value_counts().to_string())

total_liabilities_derived     203
equity_includes_nci            30
gross_profit_derived           63
inventory_assumed_zero         99
cash_includes_restricted       24
net_income_from_profitloss     41
low_revenue                   117

Interest expense source:


interest_source
reported           589
zero_no_debt       219
cash_paid_proxy    207
not_available       53
companyfacts        17


## Step 6: Drop rows missing a core item

Core items: total assets, equity, total liabilities, revenue, net income (hard rule 5). Statement values are never filled with means or invented numbers; a company-year without them cannot produce the main ratios or the Z' score.

In [9]:
clean, dropped = pp.drop_missing_core(statements)
steps.append(("After dropping rows missing core items", len(clean), clean["cik"].nunique()))
print(f"Rows dropped: {len(dropped)}")
print("Missing core item among dropped rows (a row can miss several):")
print(dropped[pp.CORE_ITEMS].isna().sum().to_string())
print()
print("Dropped rows by label year:", dropped["label_year"].value_counts().sort_index().to_dict())
print("Case companies among dropped rows:", dropped["cik"].isin(config.CASE_COMPANIES).sum())

Rows dropped: 57
Missing core item among dropped rows (a row can miss several):
total_assets          1
equity                7
total_liabilities     1
revenue              38
net_income           28

Dropped rows by label year: {2014: 10, 2015: 7, 2016: 7, 2017: 5, 2018: 4, 2019: 4, 2020: 5, 2021: 3, 2022: 5, 2023: 2, 2024: 3, 2025: 2}
Case companies among dropped rows: 0


In [10]:
# Every dropped company-year with the reason, for the data quality log.
drop_log = dropped[["cik", "name", "label_year", "period", "form"]].copy()
drop_log["missing"] = dropped[pp.CORE_ITEMS].isna().apply(lambda r: ", ".join(r.index[r]), axis=1)
no_usd_log = no_usd[["cik", "name", "label_year", "period", "form"]].assign(missing="no USD values (reports in another currency)")
drop_log = pd.concat([drop_log, no_usd_log]).sort_values(["name", "label_year"])
drop_log.to_csv(config.INTERIM_DIR / "dropped_rows.csv", index=False)
print("Saved", config.INTERIM_DIR / "dropped_rows.csv", f"({len(drop_log)} rows)")
drop_log.head(15)

Saved G:\DJSCE\Fourth Year\FADS\Mini Proj\data\interim\dropped_rows.csv (59 rows)


,cik,name,label_year,period,form,missing
0001104659-21-148729,1813658,ACE CONVERGENCE ACQUISITION CORP.,2020,2020-12-31,10-K/A,"revenue, net_income"
0001410578-22-000335,1813658,ACE CONVERGENCE ACQUISITION CORP.,2021,2021-12-31,10-K,revenue
0001213900-22-059543,1828805,"AELUMA, INC.",2022,2022-06-30,10-K,revenue
0001683168-17-000713,1420520,ATOMERA INC,2016,2016-12-31,10-K,revenue
0000950170-25-053870,1871638,"BLAIZE HOLDINGS, INC.",2024,2024-12-31,10-K,revenue
0001483798-20-000045,1191334,CHUN CAN CAPITAL GROUP,2014,2014-12-31,10-K,"equity, revenue, net_income"
0000717588-20-000004,1191334,CHUN CAN CAPITAL GROUP,2015,2015-12-31,10-K,"equity, revenue, net_income"
0000717588-20-000012,1191334,CHUN CAN CAPITAL GROUP,2016,2016-12-31,10-K,"equity, revenue, net_income"
0000717588-20-000020,1191334,CHUN CAN CAPITAL GROUP,2017,2017-12-31,10-K,"equity, revenue, net_income"
0000717588-20-000028,1191334,CHUN CAN CAPITAL GROUP,2018,2018-12-31,10-K,"equity, revenue, net_income"


## Step 7: Sanity checks (flags only; nothing is dropped here)

- Balance: assets ≈ liabilities + temporary equity + total equity, within 1% (only informative where liabilities were reported, not derived).
- Total assets and current liabilities positive; revenue positive.
- Year-on-year jump: total assets or revenue changing by more than 10× versus the previous year, which usually signals a scale error.

In [11]:
clean = pp.sanity_checks(clean)
checks = ["balance_ok", "assets_positive", "current_liabilities_positive", "revenue_positive"]
summary = pd.DataFrame({"rows_failing": (~clean[checks]).sum(), "of_which_input_missing": [
    0, 0, clean["current_liabilities"].isna().sum(), 0]})
summary.loc["yoy_jump"] = [clean["yoy_jump"].sum(), 0]
summary

,rows_failing,of_which_input_missing
balance_ok,6,0
assets_positive,0,0
current_liabilities_positive,1,1
revenue_positive,9,0
yoy_jump,11,0


In [12]:
# Balance failures: all have total liabilities reported (not derived), so the gap is real.
b = clean[~clean["balance_ok"]]
total_eq = b["equity_incl_nci"].fillna(b["equity"] + b["noncontrolling_interest"].fillna(0))
gap = b["total_assets"] - b["total_liabilities"] - b["temporary_equity"].fillna(0) - total_eq
b.assign(gap=gap, gap_pct=gap / b["total_assets"] * 100)[
    ["name", "label_year", "total_assets", "total_liabilities", "temporary_equity", "equity", "equity_incl_nci", "gap", "gap_pct", "total_liabilities_derived"]]

,name,label_year,total_assets,total_liabilities,temporary_equity,equity,equity_incl_nci,gap,gap_pct,total_liabilities_derived
adsh,,,,,,,,,,
0001493152-26-016338,KOPIN CORP,2025,108.4,36.7,NaN,64.1,64.1,7.6,7.0,False
0001654954-25-011130,LIGHTPATH TECHNOLOGIES INC,2025,81.5,31.6,NaN,15.6,NaN,34.2,42.0,False
0001045810-16-000205,NVIDIA CORP,2015,"7,370.0","2,814.0",NaN,"4,469.0",NaN,87.0,1.2,False
0001437749-25-009538,"GUERRILLA RF, INC.",2024,24.9,17.1,NaN,-12.2,NaN,20.0,80.4,False
0001437749-26-009848,"GUERRILLA RF, INC.",2025,19.5,17.7,NaN,-18.2,NaN,20.0,102.7,False
0001992243-26-000007,T1 ENERGY INC.,2025,"1,372.1","1,050.2",NaN,250.4,250.4,71.5,5.2,False


In [13]:
print("Year-on-year jumps (> 10x up or down):")
display(clean[clean["yoy_jump"]][["name", "label_year", "total_assets", "revenue"]])
print("Revenue zero or negative (kept, flagged low_revenue):")
display(clean[~clean["revenue_positive"]][["name", "label_year", "revenue", "low_revenue"]])

Year-on-year jumps (> 10x up or down):


,name,label_year,total_assets,revenue
adsh,,,,
0001445866-16-001799,"LIVING 3D HOLDINGS, INC.",2015,0.0,0.0
0001445866-17-000462,"LIVING 3D HOLDINGS, INC.",2016,0.0,0.0
0001445866-18-000394,"LIVING 3D HOLDINGS, INC.",2017,0.2,0.0
0001040161-26-000019,"PIXELWORKS, INC",2025,51.3,0.7
0000950170-25-047692,"ASCENT SOLAR TECHNOLOGIES, INC.",2024,7.1,0.0
0001295345-16-000435,SOLARFLEX CORP,2015,0.3,0.0
0001493152-18-005531,KINERJAPAY CORP.,2017,0.5,-0.1
0001493152-19-005809,KINERJAPAY CORP.,2018,1.1,2.8
0001579910-17-000006,RESONANT INC,2016,13.2,0.3


Revenue zero or negative (kept, flagged low_revenue):


,name,label_year,revenue,low_revenue
adsh,,,,
0001445866-17-000462,"LIVING 3D HOLDINGS, INC.",2016,0.0,True
0001415889-16-006045,"METASTAT, INC.",2015,0.0,True
0001415889-17-000875,"METASTAT, INC.",2016,0.0,True
0001295345-15-000057,SOLARFLEX CORP,2014,0.0,True
0001295345-16-000435,SOLARFLEX CORP,2015,0.0,True
0001295345-17-000103,KINERJAPAY CORP.,2016,0.0,True
0001493152-18-005531,KINERJAPAY CORP.,2017,-0.1,True
0001579910-16-000020,RESONANT INC,2015,0.0,True
0001504412-16-000227,"ROID GROUP, INC.",2015,0.0,True


## Step 8: Before-and-after summary

In [14]:
steps_df = pd.DataFrame(steps, columns=["step", "rows", "companies"])
steps_df["rows_removed"] = (-steps_df["rows"].diff()).fillna(0).astype(int)
steps_df.to_csv(config.PROCESSED_DIR / "preprocessing_steps.csv", index=False)
steps_df

,step,rows,companies,rows_removed
0,Phase 1 annual filings (all label years),1128,167,0
1,Label years 2014-2025,1114,167,14
2,Deduplicated (latest filing per company-period),1087,167,27
3,With USD statement values,1085,166,2
4,After dropping rows missing core items,1028,161,57


In [15]:
by_year = clean.groupby("label_year").agg(companies=("cik", "nunique"), low_revenue=("low_revenue", "sum"))
print("One row per company per label year:", not clean.duplicated(["cik", "label_year"]).any())
by_year.T

One row per company per label year: True


label_year,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
companies,103,93,83,86,85,83,81,89,83,81,81,80
low_revenue,7,11,6,9,10,11,10,13,10,8,10,8


## Step 9: The four case companies

They carry the report, so every core item must be present in every year. Values in USD millions.

In [16]:
case = clean[clean["cik"].isin(config.CASE_COMPANIES)].copy()
case["company"] = case["cik"].map(config.CASE_COMPANIES)
core_check = case.groupby("company")[pp.CORE_ITEMS + ["current_assets", "current_liabilities", "cash", "inventory",
                                                     "retained_earnings", "operating_income", "interest_expense"]].apply(lambda g: g.isna().sum())
print("Years per company:", case.groupby("company").size().to_dict())
print("Missing values per item (should all be 0):")
core_check

Years per company: {'AMD': 12, 'Intel': 12, 'Nvidia': 12, 'Wolfspeed': 12}
Missing values per item (should all be 0):


,total_assets,equity,total_liabilities,revenue,net_income,current_assets,current_liabilities,cash,inventory,retained_earnings,operating_income,interest_expense
company,,,,,,,,,,,,
AMD,0,0,0,0,0,0,0,0,0,0,0,0
Intel,0,0,0,0,0,0,0,0,0,0,0,0
Nvidia,0,0,0,0,0,0,0,0,0,0,0,0
Wolfspeed,0,0,0,0,0,0,0,0,0,0,0,0


In [17]:
show = ["label_year", "period", "revenue", "gross_profit", "operating_income", "interest_expense", "interest_source",
        "net_income", "total_assets", "total_liabilities", "equity", "current_assets", "current_liabilities", "cash", "inventory", "retained_earnings"]
for company in config.CASE_COMPANIES.values():
    print(f"=== {company}")
    display(case[case["company"] == company][show].set_index("label_year"))

=== Nvidia


,period,revenue,gross_profit,operating_income,interest_expense,interest_source,net_income,total_assets,total_liabilities,equity,current_assets,current_liabilities,cash,inventory,retained_earnings
label_year,,,,,,,,,,,,,,,
2014,2015-01-31,"4,681.5","2,599.5",759.0,46.1,reported,630.6,"7,201.4","2,783.4","4,418.0","5,713.3",896.0,496.7,482.9,"3,948.9"
2015,2016-01-31,"5,010.0","2,811.0",747.0,47.0,reported,614.0,"7,370.0","2,814.0","4,469.0","6,053.0","2,351.0",596.0,418.0,"4,350.0"
2016,2017-01-31,"6,910.0","4,063.0","1,934.0",58.0,reported,"1,666.0","9,841.0","4,048.0","5,762.0","8,536.0","1,788.0","1,766.0",794.0,"6,108.0"
2017,2018-01-31,"9,714.0","5,822.0","3,210.0",61.0,reported,"3,047.0","11,241.0","3,770.0","7,471.0","9,255.0","1,153.0","4,002.0",796.0,"8,787.0"
2018,2019-01-31,"11,716.0","7,171.0","3,804.0",58.0,reported,"4,141.0","13,292.0","3,950.0","9,342.0","10,557.0","1,329.0",782.0,"1,575.0","12,565.0"
2019,2020-01-31,"10,918.0","6,768.0","2,846.0",52.0,reported,"2,796.0","17,315.0","5,111.0","12,204.0","13,690.0","1,784.0","10,896.0",979.0,"14,971.0"
2020,2021-01-31,"16,675.0","10,396.0","4,532.0",184.0,reported,"4,332.0","28,791.0","11,898.0","16,893.0","16,055.0","3,925.0",847.0,"1,826.0","18,908.0"
2021,2022-01-31,"26,914.0","17,475.0","10,041.0",236.0,reported,"9,752.0","44,187.0","17,575.0","26,612.0","28,829.0","4,335.0","1,990.0","2,605.0","16,235.0"
2022,2023-01-31,"26,974.0","15,356.0","4,224.0",262.0,reported,"4,368.0","41,182.0","19,081.0","22,101.0","23,073.0","6,563.0","3,389.0","5,159.0","10,171.0"


=== AMD


,period,revenue,gross_profit,operating_income,interest_expense,interest_source,net_income,total_assets,total_liabilities,equity,current_assets,current_liabilities,cash,inventory,retained_earnings
label_year,,,,,,,,,,,,,,,
2014,2014-12-31,"5,506.0","1,839.0",-155.0,177.0,reported,-403.0,"3,767.0","3,580.0",187.0,"2,736.0","1,440.0",805.0,685.0,"-6,646.0"
2015,2015-12-31,"3,991.0","1,080.0",-481.0,160.0,reported,-660.0,"3,109.0","3,521.0",-412.0,"2,320.0","1,403.0",785.0,678.0,"-7,306.0"
2016,2016-12-31,"4,272.0",998.0,-372.0,156.0,reported,-497.0,"3,321.0","2,905.0",416.0,"2,530.0","1,346.0","1,264.0",751.0,"-7,803.0"
2017,2017-12-31,"5,329.0","1,823.0",204.0,126.0,reported,43.0,"3,540.0","2,929.0",611.0,"2,622.0","1,486.0","1,185.0",739.0,"-7,760.0"
2018,2018-12-31,"6,475.0","2,447.0",451.0,121.0,reported,337.0,"4,556.0","3,290.0","1,266.0","3,540.0","1,984.0","1,078.0",845.0,"-7,436.0"
2019,2019-12-31,"6,731.0","2,868.0",631.0,94.0,reported,341.0,"6,028.0","3,201.0","2,827.0","4,597.0","2,359.0","1,466.0",982.0,"-7,095.0"
2020,2020-12-31,"9,763.0","4,347.0","1,369.0",47.0,reported,"2,490.0","8,962.0","3,125.0","5,837.0","6,143.0","2,417.0","1,595.0","1,399.0","-4,605.0"
2021,2021-12-31,"16,434.0","7,929.0","3,648.0",34.0,reported,"3,162.0","12,419.0","4,922.0","7,497.0","8,583.0","4,240.0","2,535.0","1,955.0","-1,451.0"
2022,2022-12-31,"23,601.0","10,603.0","1,264.0",88.0,reported,"1,320.0","67,580.0","12,830.0","54,750.0","15,019.0","6,369.0","4,835.0","3,771.0",-131.0


=== Intel


,period,revenue,gross_profit,operating_income,interest_expense,interest_source,net_income,total_assets,total_liabilities,equity,current_assets,current_liabilities,cash,inventory,retained_earnings
label_year,,,,,,,,,,,,,,,
2014,2014-12-31,"55,870.0","35,609.0","15,347.0",192.0,companyfacts,"11,704.0","91,956.0","35,179.0","55,865.0","27,730.0","16,019.0","2,561.0","4,273.0","33,418.0"
2015,2015-12-31,"55,355.0","34,679.0","14,002.0",337.0,companyfacts,"11,420.0","103,065.0","41,083.0","61,085.0","40,356.0","15,667.0","15,308.0","5,167.0","37,614.0"
2016,2016-12-31,"59,387.0","36,191.0","12,874.0",733.0,companyfacts,"10,316.0","113,327.0","46,219.0","66,226.0","35,508.0","20,302.0","5,560.0","5,553.0","40,747.0"
2017,2017-12-31,"62,761.0","39,069.0","17,936.0",646.0,companyfacts,"9,601.0","123,249.0","53,364.0","69,019.0","29,500.0","17,421.0","3,433.0","6,983.0","42,083.0"
2018,2018-12-31,"70,848.0","43,737.0","23,316.0",468.0,companyfacts,"21,053.0","127,963.0","52,981.0","74,563.0","28,787.0","16,626.0","3,019.0","7,253.0","50,172.0"
2019,2019-12-31,"71,965.0","42,140.0","22,035.0",489.0,companyfacts,"21,048.0","136,524.0","58,865.0","77,504.0","31,239.0","22,310.0","4,194.0","8,744.0","53,523.0"
2020,2020-12-31,"77,867.0","43,612.0","23,678.0",629.0,companyfacts,"20,899.0","153,091.0","72,053.0","81,038.0","47,249.0","24,754.0","5,865.0","8,427.0","56,233.0"
2021,2021-12-31,"79,024.0","43,815.0","19,456.0",597.0,companyfacts,"19,868.0","168,406.0","73,015.0","95,391.0","57,718.0","27,462.0","4,827.0","10,776.0","68,265.0"
2022,2022-12-31,"63,054.0","26,866.0","2,334.0",496.0,companyfacts,"8,014.0","182,103.0","78,817.0","101,423.0","50,407.0","32,155.0","11,144.0","13,224.0","70,405.0"


=== Wolfspeed


,period,revenue,gross_profit,operating_income,interest_expense,interest_source,net_income,total_assets,total_liabilities,equity,current_assets,current_liabilities,cash,inventory,retained_earnings
label_year,,,,,,,,,,,,,,,
2014,2014-06-30,"1,647.6",618.8,134.3,0.0,zero_no_debt,124.2,"3,344.4",354.2,"2,990.1","1,773.9",306.7,286.8,284.8,788.6
2015,2015-06-30,"1,632.5",475.0,-72.5,1.0,cash_paid_proxy,-64.1,"2,954.4",488.0,"2,466.4","1,308.2",254.8,139.7,280.6,174.9
2016,2016-06-30,"1,616.6",487.1,-10.5,3.1,cash_paid_proxy,-21.5,"2,766.1",398.2,"2,367.8","1,156.7",223.0,166.2,303.5,-0.6
2017,2017-06-30,"1,473.0",434.6,-18.7,3.6,cash_paid_proxy,-98.1,"2,649.9",427.1,"2,222.8","1,100.6",212.0,132.6,284.4,-202.7
2018,2018-06-30,"1,493.7",407.6,-329.1,6.1,cash_paid_proxy,-280.0,"2,637.5",565.5,"2,067.1",890.1,248.3,118.9,296.0,-482.7
2019,2019-06-30,"1,080.0",391.0,-15.9,4.0,cash_paid_proxy,-375.1,"2,816.9",775.7,"2,036.2","1,412.8",268.2,500.5,187.4,-847.5
2020,2020-06-30,903.9,248.3,-209.4,34.9,companyfacts,-191.7,"3,231.0","1,141.8","2,083.1","1,592.8",291.2,448.8,179.1,"-1,039.2"
2021,2021-06-30,525.6,164.6,-313.9,45.4,companyfacts,-523.9,"3,446.8","1,330.3","2,116.5","1,478.7",448.8,379.0,166.6,"-1,563.1"
2022,2022-06-30,746.2,249.3,-247.8,25.1,companyfacts,-200.9,"3,917.5","1,478.2","2,439.3","1,762.4",388.5,449.5,227.0,"-1,764.0"


In [18]:
# Which tag supplied each case-company item (one line per company and item, years in brackets).
case_src = sources.loc[case.index].join(case[["company", "label_year"]])
rows = []
for item in ["revenue", "cost_of_revenue", "total_liabilities", "equity", "cash", "interest_expense", "net_income"]:
    for (company, tag), g in case_src.groupby(["company", item]):
        rows.append({"company": company, "item": item, "source": tag,
                     "label_years": f"{g['label_year'].min()}-{g['label_year'].max()} ({len(g)})"})
pd.DataFrame(rows).sort_values(["company", "item", "label_years"]).reset_index(drop=True)

,company,item,source,label_years
0,AMD,cash,CashAndCashEquivalentsAtCarryingValue,2014-2025 (12)
1,AMD,cost_of_revenue,CostOfGoodsSold,2014-2017 (4)
2,AMD,cost_of_revenue,CostOfGoodsAndServicesSold,2018-2025 (8)
3,AMD,equity,StockholdersEquity,2014-2025 (12)
4,AMD,interest_expense,InterestExpense,2014-2025 (12)
5,AMD,net_income,NetIncomeLoss,2014-2025 (12)
6,AMD,revenue,SalesRevenueNet,2014-2017 (4)
7,AMD,revenue,RevenueFromContractWithCustomerExcludingAssess...,2018-2025 (8)
8,AMD,total_liabilities,derived: LiabilitiesAndStockholdersEquity - te...,2014-2025 (12)
9,Intel,cash,CashAndCashEquivalentsAtCarryingValue,2014-2025 (6)


### Wolfspeed around its 2025 Chapter 11 restructuring

The 10-K for the year ended June 2025 (label year 2025) was filed during the restructuring. The table below shows the last few years so the last reliable year can be decided before ratios are computed.

In [19]:
w = case[case["company"] == "Wolfspeed"].set_index("label_year")
w[["period", "filed", "revenue", "operating_income", "interest_expense", "net_income", "total_assets",
   "total_liabilities", "current_liabilities", "equity", "cash"]].loc[2021:]

,period,filed,revenue,operating_income,interest_expense,net_income,total_assets,total_liabilities,current_liabilities,equity,cash
label_year,,,,,,,,,,,
2021,2021-06-30,2021-08-18,525.6,-313.9,45.4,-523.9,"3,446.8","1,330.3",448.8,"2,116.5",379.0
2022,2022-06-30,2022-08-22,746.2,-247.8,25.1,-200.9,"3,917.5","1,478.2",388.5,"2,439.3",449.5
2023,2023-06-30,2023-08-23,921.9,-380.6,42.6,-329.9,"6,586.7","4,964.8",627.9,"1,621.9","1,757.0"
2024,2024-06-30,2024-08-22,807.2,-445.3,246.3,-864.2,"7,984.6","7,102.5",665.3,882.1,"1,045.9"
2025,2025-06-30,2025-08-26,757.6,"-1,329.2",315.2,"-1,609.2","6,854.4","7,301.5","7,090.0",-447.1,467.2


## Step 10: Save

In [20]:
id_cols = ["cik", "name", "label_year", "period", "form", "filed", "source_quarter"]
item_cols = list(pp.ITEMS)
flag_cols = ["interest_source", "has_debt", "inventory_assumed_zero", "total_liabilities_derived", "equity_includes_nci",
             "gross_profit_derived", "cash_includes_restricted", "net_income_from_profitloss", "low_revenue",
             "balance_ok", "assets_positive", "current_liabilities_positive", "revenue_positive", "yoy_jump"]
out = clean[id_cols + item_cols + flag_cols].rename(columns={"period": "fiscal_year_end"})
out.index.name = "adsh"
out = out.reset_index().sort_values(["cik", "label_year"])
out.to_csv(config.PROCESSED_DIR / "statements_clean.csv", index=False)

src_long = (sources.loc[clean.index].join(clean[["cik", "label_year"]])
            .reset_index().melt(id_vars=["adsh", "cik", "label_year"], var_name="item", value_name="source").dropna())
src_long.to_csv(config.PROCESSED_DIR / "tag_sources.csv", index=False)

print(f"Saved statements_clean.csv: {len(out):,} rows x {out.shape[1]} columns, {out['cik'].nunique()} companies")
print(f"Saved tag_sources.csv: {len(src_long):,} rows")

Saved statements_clean.csv: 1,028 rows x 43 columns, 161 companies
Saved tag_sources.csv: 18,270 rows
